# Soviet Rail, 1955: Max Flow and Min Cut

In 1955, Ted Harris and Frank Ross at the RAND Corporation modelled the Soviet railway network as a directed graph with 44 vertices and 105 weighted edges, and asked which links, if cut, would sever supply from the Soviet interior to its satellites. One floor away, Lester Ford and Delbert Fulkerson were writing the paper that answered their question.

This notebook builds flow networks, runs Edmonds-Karp (Ford-Fulkerson with BFS augmenting paths), recovers minimum cuts, and illustrates the max-flow / min-cut theorem.

**Accompanies the lesson:** [`/lessons/graph-theory/soviet-rail`](https://lutchet.netlify.app/lessons/graph-theory/soviet-rail)

In [1]:
import sys
sys.path.insert(0, '..')

from src.graph_theory.flow import FlowNetwork, max_flow, min_cut, max_flow_steps

## 1. A small flow network

Five nodes, six directed edges, capacities chosen so the bottleneck is obvious: everything leaving the source goes through either `s -> a` (capacity 10) or `s -> b` (capacity 5), so the max flow is at most 15.

In [2]:
net = FlowNetwork()
net.add_edge('s', 'a', 10)
net.add_edge('a', 't', 10)
net.add_edge('s', 'b', 5)
net.add_edge('b', 'c', 8)
net.add_edge('c', 'a', 4)
net.add_edge('c', 't', 9)

print('Nodes:', sorted(net.nodes))
print('Edges:', sorted(net.edges))

Nodes: ['a', 'b', 'c', 's', 't']
Edges: [('a', 't', 10.0), ('b', 'c', 8.0), ('c', 'a', 4.0), ('c', 't', 9.0), ('s', 'a', 10.0), ('s', 'b', 5.0)]


## 2. Max flow

`max_flow` returns the total flow value and a dictionary of per-edge flow. The flow on any edge never exceeds its capacity, and at every node other than the source and sink the flow in equals the flow out.

In [3]:
value, flow = max_flow(net, 's', 't')
print(f'Max flow s -> t: {value}')
print()
print('Per-edge flow:')
for (u, v), f in sorted(flow.items()):
    print(f'  {u} -> {v}: {f}')

Max flow s -> t: 15.0

Per-edge flow:
  a -> t: 10.0
  b -> c: 5.0
  c -> a: 0.0
  c -> t: 5.0
  s -> a: 10.0
  s -> b: 5.0


## 3. Min cut

By the max-flow / min-cut theorem, the value of the maximum flow equals the capacity of the minimum s-t cut. `min_cut` recovers the cut edges by doing a BFS on the final residual graph and returning the edges that cross from reachable to unreachable.

In [4]:
cut_value, cut_edges = min_cut(net, 's', 't')
print(f'Min cut value: {cut_value}')
print(f'Cut edges: {sorted(cut_edges)}')
assert cut_value == value, 'Max-flow / min-cut theorem violated'

Min cut value: 15.0
Cut edges: [('s', 'a'), ('s', 'b')]


## 4. Watch Edmonds-Karp step through

`max_flow_steps` returns one entry per augmenting path found. Each step records the path, the bottleneck along it, and the cumulative flow value afterwards. The initial step (index 0) has no path and zero flow.

In [5]:
steps = max_flow_steps(net, 's', 't')
for i, step in enumerate(steps):
    if not step.path:
        print(f'  {i}: initial state, flow=0')
    else:
        path_str = ' -> '.join(step.path)
        print(f'  {i}: push {step.bottleneck} along {path_str}, flow now {step.flow_value}')

  0: initial state, flow=0
  1: push 10.0 along s -> a -> t, flow now 10.0
  2: push 5.0 along s -> b -> c -> t, flow now 15.0


## 5. A Soviet-rail-style network

Ten cities, fifteen directed rail lines. Moscow is the source; a notional Western Frontier is the sink. Capacities are in arbitrary supply units per day. This is a stylised version of the 44-vertex graph in the declassified Harris-Ross report.

In [6]:
rail = FlowNetwork()
lines = [
    ('MOW', 'LED', 30), ('MOW', 'MSK', 40), ('MOW', 'KIE', 35),
    ('LED', 'MSK', 20), ('LED', 'BER', 25),
    ('MSK', 'WAW', 30),
    ('KIE', 'WAW', 20), ('KIE', 'BUD', 25),
    ('WAW', 'BER', 35), ('WAW', 'PRG', 25),
    ('PRG', 'VIE', 20), ('PRG', 'FRO', 30),
    ('BER', 'FRO', 40),
    ('VIE', 'FRO', 25),
    ('BUD', 'VIE', 20),
]
for u, v, c in lines:
    rail.add_edge(u, v, c)

names = {
    'MOW': 'Moscow', 'LED': 'Leningrad', 'MSK': 'Minsk', 'KIE': 'Kiev',
    'WAW': 'Warsaw', 'BER': 'Berlin', 'PRG': 'Prague', 'BUD': 'Budapest',
    'VIE': 'Vienna', 'FRO': 'Frontier',
}

rail_value, rail_flow = max_flow(rail, 'MOW', 'FRO')
rail_cut_value, rail_cut = min_cut(rail, 'MOW', 'FRO')
print(f'Max supply rate, Moscow to Frontier: {rail_value}')
print(f'Min cut value:                       {rail_cut_value}')
print()
print('Cut edges (the bottleneck):')
for u, v in sorted(rail_cut):
    print(f'  {names[u]:10s} -> {names[v]:10s}  capacity {rail.capacity(u, v)}')

Max supply rate, Moscow to Frontier: 85.0
Min cut value:                       85.0

Cut edges (the bottleneck):
  Berlin     -> Frontier    capacity 40.0
  Budapest   -> Vienna      capacity 20.0
  Warsaw     -> Prague      capacity 25.0


## 6. Non-uniqueness

The max-flow value is unique. The set of maximum flows is not: different orders of augmenting paths can produce different per-edge allocations. The minimum cut is not unique either; several cuts can share the minimum capacity. The theorem guarantees only that the *values* match, not that either side is unique.

Below, we double the capacity of one of the cut edges. The old min cut is no longer minimum, and a new cut elsewhere becomes binding.

In [7]:
rail2 = FlowNetwork()
for u, v, c in lines:
    if (u, v) == ('BER', 'FRO'):
        c = c * 2  # relieve the Berlin-Frontier bottleneck
    rail2.add_edge(u, v, c)

v2, _ = max_flow(rail2, 'MOW', 'FRO')
cv2, cut2 = min_cut(rail2, 'MOW', 'FRO')
print(f'New max flow: {v2}   (was {rail_value})')
print(f'New min cut:  {cv2}')
print()
print('New cut edges:')
for u, v in sorted(cut2):
    print(f'  {names[u]:10s} -> {names[v]:10s}  capacity {rail2.capacity(u, v)}')

New max flow: 90.0   (was 85.0)
New min cut:  90.0

New cut edges:
  Leningrad  -> Berlin      capacity 25.0
  Moscow     -> Kiev        capacity 35.0
  Minsk      -> Warsaw      capacity 30.0


## 7. Your turn

A few things to try:

- Pick one edge in `rail` and raise its capacity until the min cut moves. Which edge unlocks the most additional flow?
- Remove one edge entirely (capacity 0). How much does the max flow drop? Is it always the same edge that matters most?
- Add a single new edge between two existing cities. When does it raise the max flow, and when is it wasted?

The next story leaves deterministic flow behind and asks a different question: if a walker on the graph takes random steps instead of following a plan, where does she end up? That is the random walk, and it is the mathematics that built PageRank and underlies modern knowledge-graph retrieval.